In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, mean_squared_error, zero_one_loss

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from RFObTr import obtr
from dataset import CallData, BootstrapSplitter
import time
import math
from obliquetree import Classifier as ObliqueClassifier, Regressor as ObliqueRegressor

In [ ]:
from sklearn.model_selection import KFold


def combin_calc(d, p):
    return math.comb(d,p)
def obtrcv_experiment(num_splits = 10, max_num_rules = 10, dataset_name = None, random_state = 111, max_pair=[5], max_depth = [2],
            n_trees = [2], l2_reg=[1e-10, 0.001, 0.1, 1, 10], save_results = True, train_size = 500):
    dataset_name = dataset_name
    dataset = CallData().call(dataset_name = dataset_name, data_format = 'df')
    X = dataset.X
    y = dataset.y
    train_size = min(train_size, len(X))

    df = pd.DataFrame(columns=['rep no', 'max rule no', 'max depth', 'num trees', 'max pair', 'test loss', 'train loss', 'test 01 loss', 'train 01 loss', 'num rules', 'rules complexity', 'model complexity', 'lambda', 'comp time (sec)'])
    elapsed_time = pd.DataFrame(columns=['dataset name', 'time (sec)'])
    split_ind = 0
    df_ind = 0
    print('regression type is '+dataset.learning_type)
    task = 'classification' if dataset.learning_type == 'logreg' else 'regression'
    splitter = BootstrapSplitter(reps=num_splits, train_size=train_size, random_state=random_state, replace=True)
    start_time = time.time()
    for train_idx, test_idx in splitter.split(X):
        x_train = X.loc[train_idx]
        y_train = y.loc[train_idx]
        x_test = X.loc[test_idx]
        y_test = y.loc[test_idx]
        kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
        print(f'----train set sample index sum: {np.sum(train_idx)}----')
        if split_ind > -4:
            for num_rules in np.arange(1, max_num_rules+1):
                best_lambda_per_rule = 0.0
                best_val_loss_per_rule = np.inf
                print(f'----repetition number {split_ind+1}----')
                print(f'----learning rule number {num_rules}----')
                for l2_ind, l2 in enumerate(l2_reg):
                    print(f'----regularisation lambda {l2}----')
                    val_loss = np.zeros((kf.get_n_splits(),))
                    kf_idx = 0

                    for train_cv_idx, val_cv_idx in kf.split(x_train):
                        x_train_cv = x_train.iloc[train_cv_idx]
                        y_train_cv = y_train.iloc[train_cv_idx]
                        x_valid_cv = x_train.iloc[val_cv_idx]
                        y_valid_cv = y_train.iloc[val_cv_idx]

                        model_rule_cv = obtr.ObliqueTreeEnsembles(task=task, n_trees=n_trees[0], max_depth=max_depth[0], n_pair=min(max_pair[0], x_train.shape[1]), sample_frac=0.7,
                                                                feature_frac=1, target_rules=num_rules, random_state=0, l2_reg=l2).fit(x_train_cv, y_train_cv)
                        if dataset.learning_type == 'linreg':
                            val_loss[kf_idx] = mean_squared_error(y_valid_cv, model_rule_cv.predict(x_valid_cv))
                        else:
                            val_loss[kf_idx] = log_loss(y_valid_cv, model_rule_cv.predict_proba(x_valid_cv))
                        
                        kf_idx += 1

                    avg_val_loss = np.mean(val_loss)
                    if avg_val_loss < best_val_loss_per_rule:
                        best_val_loss_per_rule = avg_val_loss
                        best_lambda_per_rule = l2

                start_time_one_iteration = time.time()
                model_rule = obtr.ObliqueTreeEnsembles(task=task, n_trees=n_trees[0], max_depth=max_depth[0], n_pair=min(max_pair[0], x_train.shape[1]), sample_frac=0.7,    
                                                        feature_frac=1, target_rules=num_rules, random_state=0, l2_reg=best_lambda_per_rule).fit(x_train, y_train)
                end_time_one_iteration = time.time()
                if dataset.learning_type == 'linreg':
                    test_loss = mean_squared_error(y_test, model_rule.predict(x_test))
                    train_loss = mean_squared_error(y_train, model_rule.predict(x_train))
                    train_01loss = 0
                    test_01loss = 0
                else:
                    test_loss = log_loss(y_test, model_rule.predict_proba(x_test))
                    train_loss = log_loss(y_train, model_rule.predict_proba(x_train))
                    train_01loss = zero_one_loss(y_train, model_rule.predict(x_train))
                    test_01loss = zero_one_loss(y_test, model_rule.predict(x_test))

                df.loc[df_ind, 'rep no'] = split_ind+1
                df.loc[df_ind,'max rule no'] = num_rules
                df.loc[df_ind,'test loss'] = test_loss
                df.loc[df_ind,'train loss'] = train_loss
                df.loc[df_ind,'train 01 loss'] = train_01loss
                df.loc[df_ind,'test 01 loss'] = test_01loss
                df.loc[df_ind,'num rules'] = model_rule.num_of_rules()
                df.loc[df_ind,'rules complexity'] = model_rule.rules_complexity()
                df.loc[df_ind,'model complexity'] = model_rule.model_complexity()
                df.loc[df_ind, 'comp time (sec)'] = end_time_one_iteration - start_time_one_iteration
                df.loc[df_ind, 'max pair'] = min(max_pair[0], x_train.shape[1])
                df.loc[df_ind, 'num trees'] = n_trees[0]
                df.loc[df_ind, 'max depth'] = max_depth[0]
                df.loc[df_ind, 'lambda'] = model_rule.l2_reg_

                df_ind += 1
            df.to_excel('experiments_results/3000_samples_15_rules/obtreecv_3000_samples_mixed_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        split_ind += 1
    end_time = time.time()
    elapsed_time.loc[0,'time (sec)'] = end_time - start_time
    elapsed_time.loc[0,'dataset name'] = dataset_name
    if save_results is True:
        df.to_excel('experiments_results/3000_samples_15_rules/obtreecv_3000_samples_mixed_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        with pd.ExcelWriter('experiments_results/3000_samples_15_rules/obtreecv_3000_samples_mixed_'+dataset_name+'.xlsx', engine='openpyxl', mode='a') as writer:
            elapsed_time.to_excel(writer, sheet_name='CalcTime', index=False)
        
    return df

In [ ]:
def obtr_experiment(num_splits = 10, max_num_rules = 10, dataset_name = None, random_state = 111, max_pair=[5], max_depth = [2],
            n_trees = [2], l2_reg=[1e-10, 0.001, 0.1, 1, 10], save_results = True, train_size = 500):
    dataset_name = dataset_name
    dataset = CallData().call(dataset_name = dataset_name, data_format = 'df')
    X = dataset.X
    y = dataset.y
    train_size = min(train_size, len(X))

    df = pd.DataFrame(columns=['rep no', 'max rule no', 'max depth', 'num trees', 'max pair', 'test loss', 'train loss', 'test 01 loss', 'train 01 loss', 'num rules', 'rules complexity', 'model complexity', 'lambda', 'comp time (sec)'])
    elapsed_time = pd.DataFrame(columns=['dataset name', 'time (sec)'])
    split_ind = 0
    df_ind = 0
    print('regression type is '+dataset.learning_type)
    task = 'classification' if dataset.learning_type == 'logreg' else 'regression'
    splitter = BootstrapSplitter(reps=num_splits, train_size=train_size, random_state=random_state, replace=True)
    start_time = time.time()
    for train_idx, test_idx in splitter.split(X):
        x_train = X.loc[train_idx]
        y_train = y.loc[train_idx]
        x_test = X.loc[test_idx]
        y_test = y.loc[test_idx]
        print(f'----train set sample index sum: {np.sum(train_idx)}----')
        if split_ind > -4:
            for num_rules in np.arange(1, max_num_rules+1):
                best_lambda_per_rule = 0.0
                best_val_loss_per_rule = np.inf
                print(f'----repetition number {split_ind+1}----')
                print(f'----learning rule number {num_rules}----')
                for l2_ind, l2 in enumerate(l2_reg):
                    print(f'----regularisation lambda {l2}----')
                    start_time_one_iteration = time.time()
                    model_rule = obtr.ObliqueTreeEnsembles(task=task, n_trees=n_trees[0], max_depth=max_depth[0], n_pair=min(max_pair[0], x_train.shape[1]), sample_frac=0.7,    
                                                            feature_frac=1, target_rules=num_rules, random_state=0, l2_reg=l2).fit(x_train, y_train)
                    end_time_one_iteration = time.time()
                    if dataset.learning_type == 'linreg':
                        test_loss = mean_squared_error(y_test, model_rule.predict(x_test))
                        train_loss = mean_squared_error(y_train, model_rule.predict(x_train))
                        train_01loss = 0
                        test_01loss = 0
                    else:
                        test_loss = log_loss(y_test, model_rule.predict_proba(x_test))
                        train_loss = log_loss(y_train, model_rule.predict_proba(x_train))
                        train_01loss = zero_one_loss(y_train, model_rule.predict(x_train))
                        test_01loss = zero_one_loss(y_test, model_rule.predict(x_test))

                    df.loc[df_ind, 'rep no'] = split_ind+1
                    df.loc[df_ind,'max rule no'] = num_rules
                    df.loc[df_ind,'test loss'] = test_loss
                    df.loc[df_ind,'train loss'] = train_loss
                    df.loc[df_ind,'train 01 loss'] = train_01loss
                    df.loc[df_ind,'test 01 loss'] = test_01loss
                    df.loc[df_ind,'num rules'] = model_rule.num_of_rules()
                    df.loc[df_ind,'rules complexity'] = model_rule.rules_complexity()
                    df.loc[df_ind,'model complexity'] = model_rule.model_complexity()
                    df.loc[df_ind, 'comp time (sec)'] = end_time_one_iteration - start_time_one_iteration
                    df.loc[df_ind, 'max pair'] = min(max_pair[0], x_train.shape[1])
                    df.loc[df_ind, 'num trees'] = n_trees[0]
                    df.loc[df_ind, 'max depth'] = max_depth[0]
                    df.loc[df_ind, 'lambda'] = model_rule.l2_reg_

                    df_ind += 1
            df.to_excel('experiments_results/3000_samples_15_rules/obtree_3000_samples_mixed_repeat_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        split_ind += 1
    end_time = time.time()
    elapsed_time.loc[0,'time (sec)'] = end_time - start_time
    elapsed_time.loc[0,'dataset name'] = dataset_name
    if save_results is True:
        df.to_excel('experiments_results/3000_samples_15_rules/obtree_3000_samples_mixed_repeat_'+dataset_name+'.xlsx', sheet_name='Sheet1', index=False)
        with pd.ExcelWriter('experiments_results/3000_samples_15_rules/obtree_3000_samples_mixed_repeat_'+dataset_name+'.xlsx', engine='openpyxl', mode='a') as writer:
            elapsed_time.to_excel(writer, sheet_name='CalcTime', index=False)
        
    return df

In [ ]:
dataset_name_list = ['red_wine', 'diabetes', 'make_friedman1', 'used_car_price', 'liver', 'banknote', 'make_friedman2', 'make_friedman3', 
                     'iris','breast_cancer', 'magic04','voice',  'adult',  'california_housing_price']
num_splits = 15
max_num_rules = 15
max_pair = [5]
max_depth = [3]
num_trees = [5]
train_size = 3000
l2_reg = [1e-10, 0.001, 0.1, 1, 10]

In [ ]:
for ds in dataset_name_list:
    print(f'################################### {ds} ##################################')
    df_cv = obtrcv_experiment(num_splits = num_splits, max_num_rules = max_num_rules, dataset_name = ds, random_state = 111, max_pair=max_pair, 
                         max_depth = max_depth, n_trees = num_trees, l2_reg=l2_reg, save_results = False, train_size=train_size)
    # df = obtr_experiment(num_splits = num_splits, max_num_rules = max_num_rules, dataset_name = ds, random_state = 111, max_pair=max_pair, 
    #                      max_depth = max_depth, n_trees = num_trees, l2_reg=l2_reg, save_results = False, train_size=train_size)